# Setup 

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


import os
DRIVE_ROOT = '/content/drive/MyDrive/stylessp'
CKPT_DIR   = f'{DRIVE_ROOT}/checkpoints'
HF_CACHE   = f'{DRIVE_ROOT}/hf_cache'
REPO_DIR   = '/content/StyleSSP'

for d in [DRIVE_ROOT, CKPT_DIR, HF_CACHE]:
    os.makedirs(d, exist_ok=True)


os.environ['HF_HOME'] = HF_CACHE

fresh copy for every session, and symlink to drive folder

In [ ]:


!rm -rf {REPO_DIR}
!git clone https://github.com/Michael-Boat-2/StyleSSP {REPO_DIR}
%cd {REPO_DIR}
!git log --oneline -1

!ln -sfn {CKPT_DIR} {REPO_DIR}/checkpoints

Installing Pinned Libraries

In [ ]:

!pip install -q diffusers==0.30.0 transformers==4.44.0 accelerate==0.33.0 huggingface-hub==0.24.5 \
    controlnet-aux==0.0.9 timm==0.6.7 pyrallis==0.3.1 lpips==0.1.4 einops ftfy
!pip install -q git+https://github.com/openai/CLIP.git

import torch, diffusers, transformers, numpy
print('torch', torch.__version__, '| diffusers', diffusers.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)

Downloading Checkpoints 

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download

# IP-Adapter:  SDXL weights file for style injection during sampling
snapshot_download('h94/IP-Adapter', local_dir=f'{CKPT_DIR}/IP-Adapter',
                  allow_patterns=['sdxl_models/ip-adapter_sdxl_vit-h.safetensors'])

# ControlNets (control_type = 'tile_canny' in config.py) : content preservation during sampling
snapshot_download('xinsir/controlnet-tile-sdxl-1.0', local_dir=f'{CKPT_DIR}/controlnet-tile-sdxl-1.0',
                  allow_patterns=['config.json', '*.safetensors'])
snapshot_download('TheMistoAI/MistoLine', local_dir=f'{CKPT_DIR}/MistoLine',
                  allow_patterns=['config.json', '*.fp16.safetensors'])

# IP-Adapter-Instruct: the content/style extractor behind negative guidance
hf_hub_download('CiaraRowles/IP-Adapter-Instruct', 'ip-adapter-instruct-sdxl.bin',
                local_dir=f'{CKPT_DIR}/models')

!du -sh {CKPT_DIR}/*

Recording the used environment for the reproduction

In [ ]:
!mkdir -p {REPO_DIR}/repro/env
!python --version > {REPO_DIR}/repro/env/colab_freeze.txt
!pip freeze >> {REPO_DIR}/repro/env/colab_freeze.txt
!cp {REPO_DIR}/repro/env/colab_freeze.txt {DRIVE_ROOT}/colab_freeze.txt
!head -3 {REPO_DIR}/repro/env/colab_freeze.txt

In [ ]:
# test pair for our config

CONTENT = f'{DRIVE_ROOT}/test_images/content.jpg'
STYLE   = f'{DRIVE_ROOT}/test_images/style.jpg'

!sed -i "s|style_image_dir = .*|style_image_dir = '{STYLE}'|" src/config.py
!sed -i "s|content_image_dir = .*|content_image_dir = '{CONTENT}'|" src/config.py
!grep -n "image_dir" src/config.py

In [ ]:
%%time
!python infer_style.py

In [ ]:
from IPython.display import display
from PIL import Image
import glob, shutil

for p in sorted(glob.glob('results/*.png')):
    print(p)
    display(Image.open(p).resize((512, 512)))
    shutil.copy(p, f'{DRIVE_ROOT}/')